# FOLIO LMS: Instance Record Count

A minimal notebook that gets a count of Instance records. Start with this to ensure you have called the login notebook successfully and authenticated against your FOLIO tenant. 

In [4]:
import pandas as pd
import requests
from datetime import datetime

pd.set_option('display.max_columns', None)

## 1. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [5]:
%run folio_auth.ipynb

Login succeeded. Token retrieved.


## 2. Get ongoing irders for current fiscal year
### - List fiscal years
### - Get ongoing orders between date1 and date1 of current FY
## 3. Create subset of orders without invoices

In [6]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

In [ ]:
fys_raw = fetch_all_records(
    "/finance/fiscal-years",
    records_key="fiscalYears"
) 
fys_df = pd.DataFrame(fys_raw)
today = datetime.today().strftime("%Y-%m-%d")
cur_fy_df = fys_df[(fys_df['periodStart'] <= today) & (fys_df['periodEnd'] >= today)]
styled_fy = cur_fy_df.style.set_properties(
  **{'border': '1px solid black', 'background-color': 'lightgrey'}  
)
print(f"Fiscal Years that encompass {today}")
cur_fy_df[
    ['id', 'name', 'periodStart', 'periodEnd']
].style.set_properties(
    **{'border': '1px solid black'}
)

Fiscal Years for 2026-08-21


,id,name,periodStart,periodEnd
7,d01d3d59-a8e7-4b58-a485-5146f5747dbb,Fiscal Year 2027,2026-07-01T00:00:00.000+00:00,2027-06-30T00:00:00.000+00:00
8,ab5d32df-99ec-4487-86ea-e4203143c8d3,Calendar Year 2026,2026-01-01T00:00:00.000+00:00,2026-12-31T00:00:00.000+00:00
